# Module 3 • Classical Natural Language Processing

# Lesson 14 • Stop Words, N-Grams, and Feature Engineering

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 90–120 minutes

---

## Scope

This lesson explains how classical NLP systems select and construct useful
features. It focuses on stop-word policies, word and character n-grams,
document-frequency filtering, binary and count features, vocabulary control,
feature inspection, feature selection, multilingual considerations, and
leakage-safe pipelines.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define stop words and explain why no universal list is correct for every task;
- explain how stop-word removal may help or harm performance;
- preserve negation and other task-relevant function words;
- generate unigram, bigram, and trigram features;
- distinguish word n-grams from character n-grams;
- explain document-frequency filtering;
- compare binary, count, and weighted feature representations;
- inspect learned vocabularies and sparse matrices;
- use feature selection inside a classification pipeline;
- avoid vocabulary and feature-selection leakage;
- design task-aware English and Arabic feature policies;
- evaluate feature choices using cross-validation and error analysis.

## Table of Contents

1. What Is Feature Engineering?
2. Stop Words
3. When Stop-Word Removal Helps
4. When Stop-Word Removal Harms
5. Task-Specific Stop-Word Lists
6. N-Grams
7. Unigrams, Bigrams, and Trigrams
8. Word N-Grams
9. Character N-Grams
10. Sparse Feature Matrices
11. Binary and Count Features
12. Vocabulary Pruning
13. Feature Inspection
14. Feature Selection
15. Classification Pipelines
16. Avoiding Data Leakage
17. Multilingual and Arabic Considerations
18. Evaluation and Error Analysis
19. Knowledge Check
20. Exercises
21. Summary and Next Lesson

# 1. What Is Feature Engineering?

**Feature engineering** converts raw text into measurable signals that a model
can use.

Classical NLP features may include:

- word counts;
- binary word presence;
- word n-grams;
- character n-grams;
- punctuation counts;
- document length;
- capitalization;
- lexical categories;
- domain-specific dictionaries;
- syntactic or semantic labels.

The usefulness of a feature depends on the task, dataset, and model.

In [ ]:
import pandas as pd

feature_examples = pd.DataFrame(
    [
        ("Unigram count", "number of times each word occurs"),
        ("Bigram presence", "whether a two-word sequence occurs"),
        ("Character n-gram", "short character sequence"),
        ("Punctuation count", "number of !, ?, or other symbols"),
        ("Document length", "number of words or characters"),
        ("Domain lexicon", "presence of specialized terms"),
    ],
    columns=["Feature", "Description"],
)

feature_examples

> **Key Idea**
>
> A useful feature preserves information relevant to the prediction while
> reducing irrelevant variation.

# 2. Stop Words

**Stop words** are high-frequency words that some pipelines remove before
feature extraction.

Common English examples include:

```text
the, a, an, is, of, and, to
```

However, stop words may carry grammatical, stylistic, or semantic information.

In [ ]:
sample_stop_words = {
    "the", "a", "an", "is", "of", "and", "to", "in",
}

sentence = "the model is accurate and useful"

tokens = sentence.split()
filtered = [
    token for token in tokens
    if token not in sample_stop_words
]

print("Original:", tokens)
print("Filtered:", filtered)

Stop-word removal reduces vocabulary and may simplify sparse models, but the
decision should be validated rather than assumed.

# 3. When Stop-Word Removal Helps

Stop-word removal may help when:

- the model uses sparse count features;
- the task focuses on broad topical content;
- frequent function words dominate the feature space;
- documents are long;
- computational resources are limited;
- retrieval should emphasize content words.

In [ ]:
documents = [
    "the model analyzes the document",
    "the system analyzes the data",
    "the model predicts the category",
]

raw_vocabulary = {
    token
    for document in documents
    for token in document.split()
}

filtered_vocabulary = {
    token
    for document in documents
    for token in document.split()
    if token not in sample_stop_words
}

print("Raw vocabulary:", sorted(raw_vocabulary))
print("Filtered vocabulary:", sorted(filtered_vocabulary))

# 4. When Stop-Word Removal Harms

Stop-word removal may damage:

- sentiment analysis;
- question answering;
- authorship analysis;
- machine translation;
- grammar-sensitive tasks;
- phrase retrieval;
- intent classification;
- text generation.

Example:

```text
useful
not useful
```

Removing `not` reverses the intended meaning.

In [ ]:
risky_stop_words = sample_stop_words | {"not", "no", "never"}

examples = [
    "the product is useful",
    "the product is not useful",
    "no issue was found",
    "an issue was found",
]

for text in examples:
    kept = [
        token for token in text.split()
        if token not in risky_stop_words
    ]
    print(f"{text:<30} -> {kept}")

Function words can also reveal writing style, formality, and authorship.

# 5. Task-Specific Stop-Word Lists

A strong stop-word policy should specify:

- language;
- domain;
- task;
- protected words;
- negation policy;
- version;
- evaluation evidence.

In [ ]:
BASE_STOP_WORDS = {
    "the", "a", "an", "is", "of", "and", "to", "in",
}

PROTECTED_WORDS = {
    "not", "no", "never", "without",
}

task_stop_words = BASE_STOP_WORDS - PROTECTED_WORDS

print("Task stop words:", sorted(task_stop_words))
print("Protected words:", sorted(PROTECTED_WORDS))

A domain-specific term should not be removed merely because it is frequent.
In medical records, for example, frequent terms may still be clinically
important.

# 6. N-Grams

An **n-gram** is a contiguous sequence of `n` units.

Word n-grams:

```text
natural language processing

unigrams: natural, language, processing
bigrams: natural language, language processing
trigram: natural language processing
```

In [ ]:
def generate_ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    if n <= 0:
        raise ValueError("n must be positive")

    return [
        tuple(tokens[index:index + n])
        for index in range(len(tokens) - n + 1)
    ]


tokens = "natural language processing is useful".split()

for n in [1, 2, 3]:
    print(f"{n}-grams:", generate_ngrams(tokens, n))

N-grams preserve limited local context while remaining compatible with
classical sparse models.

# 7. Unigrams, Bigrams, and Trigrams

## Unigrams

Represent individual words.

Advantages:

- simple;
- compact;
- robust with limited data.

Limitations:

- ignore local word order;
- may miss negation and phrases.

## Bigrams

Represent two-word sequences.

Examples:

```text
not good
machine learning
New York
```

## Trigrams

Represent three-word sequences.

Examples:

```text
not very good
natural language processing
```

In [ ]:
phrase = "not very useful"

unigram_features = generate_ngrams(phrase.split(), 1)
bigram_features = generate_ngrams(phrase.split(), 2)
trigram_features = generate_ngrams(phrase.split(), 3)

print("Unigrams:", unigram_features)
print("Bigrams:", bigram_features)
print("Trigrams:", trigram_features)

Larger n-grams capture more context but increase sparsity and require more data.

# 8. Word N-Grams

`CountVectorizer` can create word n-gram features.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

ngram_documents = [
    "the service was good",
    "the service was not good",
    "the support was very good",
]

word_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
)

word_matrix = word_vectorizer.fit_transform(ngram_documents)

print("Matrix shape:", word_matrix.shape)
print("Features:")
print(word_vectorizer.get_feature_names_out())

In [ ]:
word_feature_frame = pd.DataFrame(
    word_matrix.toarray(),
    columns=word_vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(len(ngram_documents))],
)

word_feature_frame

The bigram `not good` preserves a distinction that unigram features may model
less directly.

# 9. Character N-Grams

Character n-grams represent overlapping character sequences.

They can capture:

- prefixes and suffixes;
- spelling variation;
- morphology;
- partial words;
- punctuation style;
- noisy text;
- cross-word patterns.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

character_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
)

character_matrix = character_vectorizer.fit_transform(
    [
        "connect",
        "connected",
        "connection",
        "disconnect",
    ]
)

print("Character matrix shape:", character_matrix.shape)
print("First 20 character features:")
print(character_vectorizer.get_feature_names_out()[:20])

Character n-grams often support misspellings and morphologically rich
languages, but they are less interpretable than word features and can create a
large feature space.

## 9.1 Word Versus Character N-Grams

| Property | Word n-grams | Character n-grams |
|---|---|---|
| Unit | words | characters |
| Interpretability | usually high | lower |
| Spelling robustness | limited | stronger |
| Vocabulary size | can be large | can be very large |
| Morphological clues | indirect | often strong |
| Tokenizer dependence | high | lower |

# 10. Sparse Feature Matrices

Text feature matrices are usually sparse because each document contains only a
small fraction of the complete vocabulary.

In [ ]:
sparse_matrix = word_matrix

total_cells = sparse_matrix.shape[0] * sparse_matrix.shape[1]
nonzero_cells = sparse_matrix.nnz
density = nonzero_cells / total_cells

print("Rows:", sparse_matrix.shape[0])
print("Columns:", sparse_matrix.shape[1])
print("Nonzero cells:", nonzero_cells)
print(f"Density: {density:.3f}")

Sparse matrix formats store only nonzero values and their locations, making
large vocabularies computationally manageable.

# 11. Binary and Count Features

Count features record frequency.

Binary features record whether a feature is present.

In [ ]:
count_vectorizer = CountVectorizer(binary=False)
binary_vectorizer = CountVectorizer(binary=True)

count_matrix = count_vectorizer.fit_transform(
    ["good good service", "good support"]
)

binary_matrix = binary_vectorizer.fit_transform(
    ["good good service", "good support"]
)

features = count_vectorizer.get_feature_names_out()

print("Features:", features)
print("Count matrix:")
print(count_matrix.toarray())
print("Binary matrix:")
print(binary_matrix.toarray())

Binary features may work well when repeated occurrences should not increase the
feature's influence. Count features preserve frequency information.

# 12. Vocabulary Pruning

Vocabulary pruning controls feature growth.

Common parameters include:

- `min_df`: minimum document frequency;
- `max_df`: maximum document frequency;
- `max_features`: maximum vocabulary size;
- custom stop-word lists.

In [ ]:
pruning_documents = [
    "the model analyzes text",
    "the model analyzes documents",
    "the system processes text",
    "rareterm appears once",
]

full_vectorizer = CountVectorizer()
pruned_vectorizer = CountVectorizer(min_df=2)

full_vectorizer.fit(pruning_documents)
pruned_vectorizer.fit(pruning_documents)

print("Full vocabulary:")
print(full_vectorizer.get_feature_names_out())

print("\nVocabulary with min_df=2:")
print(pruned_vectorizer.get_feature_names_out())

High-frequency features may be uninformative, but `max_df` should not remove a
frequent domain term without evaluation.

# 13. Feature Inspection

Inspecting learned features helps detect:

- preprocessing mistakes;
- broken tokenization;
- data leakage;
- identifiers;
- rare noise;
- unexpected punctuation;
- mislabeled examples.

In [ ]:
inspection_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    min_df=1,
)

inspection_matrix = inspection_vectorizer.fit_transform(
    [
        "excellent service",
        "not excellent",
        "very slow service",
        "support@example.com",
    ]
)

print(inspection_vectorizer.get_feature_names_out())

The token `support@example.com` is split by the default tokenizer. This may be
appropriate or may reveal a preprocessing mismatch.

# 14. Feature Selection

Feature selection retains features that are most informative for the target.

Common approaches include:

- document-frequency thresholds;
- chi-squared selection;
- mutual information;
- model-based selection;
- regularization.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

selection_texts = [
    "excellent service",
    "helpful support",
    "fast response",
    "terrible service",
    "rude support",
    "slow response",
]

selection_labels = [
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
]

selection_vectorizer = CountVectorizer()
selection_matrix = selection_vectorizer.fit_transform(selection_texts)

selector = SelectKBest(
    score_func=chi2,
    k=4,
)

selected_matrix = selector.fit_transform(
    selection_matrix,
    selection_labels,
)

selected_features = (
    selection_vectorizer
    .get_feature_names_out()[selector.get_support()]
)

print("Selected features:", selected_features)
print("Selected matrix shape:", selected_matrix.shape)

Feature selection must be fitted only on training data. Selecting features from
the complete dataset leaks test-label information.

# 15. Classification Pipelines

Pipelines allow feature extraction, selection, and classification to be
evaluated together.

In [ ]:
classification_texts = [
    "excellent service and helpful staff",
    "the support was excellent",
    "fast and friendly response",
    "helpful customer service",
    "terrible service and rude staff",
    "the support was terrible",
    "slow and unhelpful response",
    "rude customer service",
    "excellent and fast support",
    "terrible and slow service",
    "friendly staff and useful help",
    "unhelpful staff and bad response",
]

classification_labels = [
    "positive",
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "negative",
    "positive",
    "negative",
]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

unigram_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=(1, 1),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

bigram_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

unigram_scores = cross_val_score(
    unigram_pipeline,
    classification_texts,
    classification_labels,
    cv=3,
    scoring="f1_macro",
)

bigram_scores = cross_val_score(
    bigram_pipeline,
    classification_texts,
    classification_labels,
    cv=3,
    scoring="f1_macro",
)

print("Unigram scores:", unigram_scores.round(3))
print("Bigram scores:", bigram_scores.round(3))

In [ ]:
comparison = pd.DataFrame(
    {
        "Feature configuration": [
            "Unigrams",
            "Unigrams + bigrams",
        ],
        "Mean macro F1": [
            unigram_scores.mean(),
            bigram_scores.mean(),
        ],
    }
)

comparison

The dataset is intentionally small. The purpose is to demonstrate experimental
comparison, not to establish a general conclusion.

## 15.1 Combining Word and Character Features

`FeatureUnion` can combine multiple feature spaces.

In [ ]:
from sklearn.pipeline import FeatureUnion

combined_features = FeatureUnion(
    [
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
            ),
        ),
        (
            "character",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
            ),
        ),
    ]
)

combined_matrix = combined_features.fit_transform(
    classification_texts
)

print("Combined feature matrix shape:", combined_matrix.shape)

Combined representations may improve robustness but increase memory use,
training time, and overfitting risk.

# 16. Avoiding Data Leakage

Leakage occurs when test data influences vocabulary, feature weights, feature
selection, or stop-word construction.

Unsafe workflow:

```text
fit vectorizer on all data
select features on all labels
split into train and test
```

Safer workflow:

```text
split or cross-validate
fit vectorizer and selector only on training data
apply learned transformations to validation or test data
```

Scikit-learn pipelines perform fitting inside each cross-validation fold.

In [ ]:
leakage_safe_pipeline = Pipeline(
    [
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=1,
            ),
        ),
        (
            "selector",
            SelectKBest(
                score_func=chi2,
                k=10,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

safe_scores = cross_val_score(
    leakage_safe_pipeline,
    classification_texts,
    classification_labels,
    cv=3,
    scoring="f1_macro",
)

print("Leakage-safe pipeline scores:", safe_scores.round(3))

# 17. Multilingual and Arabic Considerations

Stop-word and n-gram policies must match the language.

Challenges include:

- morphology;
- clitics;
- script;
- normalization;
- dialect;
- code-switching;
- tokenization standards;
- resource availability.

## 17.1 Arabic Stop Words

Arabic function words may appear as separate tokens or attached clitics.

Examples:

```text
و
في
من
إلى
بالكتاب
والكتاب
```

Stop-word filtering after whitespace tokenization may fail to recognize
attached conjunctions and prepositions.

In [ ]:
arabic_stop_words = {
    "في", "من", "إلى", "على", "عن", "و",
}

arabic_sentence = "وذهب الطالب إلى المدرسة"

arabic_tokens = arabic_sentence.split()
arabic_filtered = [
    token for token in arabic_tokens
    if token not in arabic_stop_words
]

print("Tokens:", arabic_tokens)
print("Filtered:", arabic_filtered)

The attached `و` remains part of `وذهب`. A segmentation policy may be needed
before stop-word filtering.

## 17.2 Arabic Character N-Grams

Character n-grams may capture partial stems, affixes, and spelling variation
without requiring perfect segmentation.

In [ ]:
arabic_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
)

arabic_char_matrix = arabic_char_vectorizer.fit_transform(
    [
        "كتاب",
        "الكتاب",
        "والكتاب",
        "كتابه",
    ]
)

print("Arabic character feature matrix shape:", arabic_char_matrix.shape)

Character features do not replace morphological analysis, but they can provide
useful robust signals for classical models.

# 18. Evaluation and Error Analysis

Feature engineering should be evaluated using:

- cross-validation;
- macro and weighted F1;
- precision and recall;
- vocabulary size;
- matrix density;
- training time;
- memory usage;
- error categories;
- robustness by language, domain, and text length.

## Common Feature-Engineering Errors

- removing negation;
- using a generic stop-word list without review;
- retaining identifiers that leak labels;
- fitting the vocabulary on the full dataset;
- selecting features using test labels;
- using n-grams too large for the dataset;
- pruning rare but important domain terms;
- mixing inconsistent tokenization policies;
- ignoring Arabic clitics;
- creating an excessively large character feature space.

In [ ]:
feature_errors = pd.DataFrame(
    [
        ("Remove 'not'", "sentiment polarity can reverse"),
        ("Fit vocabulary before split", "test information leaks"),
        ("Use trigrams on tiny data", "severe sparsity"),
        ("Keep ticket IDs", "identifier leakage"),
        ("Remove frequent medical term", "important domain signal lost"),
        ("Ignore Arabic clitics", "stop-word matching fails"),
    ],
    columns=["Decision", "Risk"],
)

feature_errors

# 19. Knowledge Check

1. What is feature engineering?
2. What is a stop word?
3. Why is there no universally correct stop-word list?
4. Why should negation often be preserved?
5. What is an n-gram?
6. How do unigrams, bigrams, and trigrams differ?
7. What are the advantages of character n-grams?
8. Why are text feature matrices sparse?
9. How do binary and count features differ?
10. What do `min_df` and `max_df` control?
11. Why should learned features be inspected?
12. What is feature-selection leakage?
13. Why should vectorization remain inside a pipeline?
14. Why do Arabic stop-word policies depend on segmentation?

# 20. Exercises

## Exercise 1 — Stop-Word Audit

Compare a generic English stop-word list with the words required by a
sentiment task. Identify unsafe removals.

## Exercise 2 — Protected Words

Create a task-specific stop-word list that preserves negation, question words,
and domain terms.

## Exercise 3 — N-Gram Generator

Extend the n-gram function to generate a selected range such as 1–3.

## Exercise 4 — Word Features

Compare unigram, bigram, and trigram vocabularies for a small corpus.

## Exercise 5 — Character Features

Compare character n-gram settings `(2, 4)`, `(3, 5)`, and `(4, 6)`.

## Exercise 6 — Vocabulary Pruning

Measure how `min_df`, `max_df`, and `max_features` change vocabulary size and
macro F1.

## Exercise 7 — Feature Selection

Apply chi-squared selection inside a pipeline and compare several values of
`k`.

## Exercise 8 — Arabic Features

Compare whitespace tokens, light segmentation, and character n-grams on Arabic
text.

## Challenge Exercises

1. Combine word and character TF-IDF features in one classifier.
2. Build a feature-audit report containing vocabulary size, density, and top
   weighted features.
3. Compare binary counts, raw counts, and TF-IDF.
4. Create language-specific stop-word policies for English and Arabic.
5. Perform nested cross-validation for n-gram and pruning choices.

# 21. Summary and Next Lesson

In this lesson:

- feature engineering converted text into model-readable signals;
- stop-word removal was treated as task-specific rather than automatic;
- negation and domain terms were preserved when necessary;
- unigrams represented individual words;
- bigrams and trigrams captured local word order;
- character n-grams captured spelling and morphological variation;
- sparse matrices stored large feature spaces efficiently;
- binary and count features represented different frequency assumptions;
- vocabulary pruning controlled dimensionality;
- feature inspection exposed noise and leakage;
- chi-squared selection retained informative features;
- pipelines prevented vocabulary and feature-selection leakage;
- Arabic feature design required attention to clitics and segmentation;
- evaluation combined predictive metrics with vocabulary, density, and error
  analysis.

## Next Lesson

**Lesson 15: Bag-of-Words and TF-IDF Representation** examines term-document
matrices, term frequency, inverse document frequency, normalization, feature
weighting, similarity, and interpretation.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Manning, C. D., Raghavan, P., & Schütze, H. *Introduction to Information Retrieval*.
- scikit-learn feature extraction and feature selection documentation.
- classical text classification and n-gram literature.
- Arabic text-classification and feature-engineering literature.